# Ghi nhớ bài học: Fine-tuning hay RAG để augment LLM?

Notebook Colab này giúp bạn **remember bài học bằng code** thay vì chỉ đọc lý thuyết.

Bài học chính:

- **Fine-tuning**: thay đổi/cập nhật model để model học một **kỹ năng, format, hành vi, hoặc task cụ thể**.
- **RAG**: không đổi model; thay vào đó, lấy thông tin liên quan từ database/tài liệu rồi nhét vào prompt để model trả lời.
- Hai cách này **không loại trừ nhau**. Nhiều hệ thống tốt dùng cả hai.

## Cách học gợi ý

1. Chạy từng cell từ trên xuống dưới.
2. Đọc phần giải thích ngắn trước mỗi đoạn code.
3. Thử sửa biến đầu vào trong các cell `TODO` để tự kiểm chứng.
4. Cuối notebook có quiz nhỏ để tự ôn.

## 1. Setup

Notebook chỉ dùng `numpy`, `pandas`, `matplotlib` để chạy dễ trên Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("default")
np.random.seed(7)

## 2. Mental model: Fine-tuning vs RAG

Hãy nhớ bằng 1 câu:

> **Fine-tuning dạy model cách hành xử. RAG đưa thêm tài liệu cho model đọc trước khi trả lời.**

Bảng dưới đây biến ý tưởng này thành các tiêu chí ra quyết định.

In [ ]:
comparison = pd.DataFrame([
    {
        "criterion": "Model có bị thay đổi không?",
        "Fine-tuning": "Có: cập nhật weight hoặc adapter LoRA/QLoRA",
        "RAG": "Không: chỉ thêm context vào prompt",
    },
    {
        "criterion": "Phù hợp khi cần...",
        "Fine-tuning": "Học task, style, format, routing, NER, sentiment",
        "RAG": "Truy cập kiến thức/tài liệu riêng hoặc hay thay đổi",
    },
    {
        "criterion": "Dữ liệu cần chuẩn bị",
        "Fine-tuning": "Dataset train/validation chất lượng cao",
        "RAG": "Tài liệu được chunk, embed, index trong vector DB",
    },
    {
        "criterion": "Rủi ro chính",
        "Fine-tuning": "Tốn pipeline, validation, có thể catastrophic forgetting",
        "RAG": "Retrieve sai chunk, không tốt cho câu hỏi cần aggregate toàn DB",
    },
    {
        "criterion": "Khi dữ liệu đổi liên tục",
        "Fine-tuning": "Có thể phải train lại định kỳ",
        "RAG": "Update index/vector DB thường dễ hơn",
    },
])
comparison

## 3. Decision helper bằng code

Hàm dưới đây là một **rule-of-thumb**. Nó không thay thế kiến trúc thật, nhưng giúp bạn nhớ logic chọn hướng:

- Nếu cần kiến thức mới/tài liệu riêng/thường xuyên thay đổi → nghiêng về **RAG**.
- Nếu cần model học format/task/hành vi ổn định → nghiêng về **Fine-tuning**.
- Nếu cần cả kiến thức ngoài lẫn hành vi đặc thù → có thể dùng **Both**.

In [ ]:
def choose_augmentation(
    needs_private_knowledge: bool,
    data_changes_often: bool,
    needs_strict_output_format: bool,
    needs_new_skill_or_behavior: bool,
    has_labeled_training_data: bool,
    question_requires_global_aggregation: bool,
):
    rag_score = 0
    ft_score = 0
    warnings = []

    if needs_private_knowledge:
        rag_score += 3
    if data_changes_often:
        rag_score += 3
    if needs_strict_output_format:
        ft_score += 2
    if needs_new_skill_or_behavior:
        ft_score += 3
    if has_labeled_training_data:
        ft_score += 2
    else:
        ft_score -= 2
    if question_requires_global_aggregation:
        warnings.append("RAG thuần không tốt cho câu hỏi cần scan/aggregate toàn bộ database.")
        rag_score -= 2

    if rag_score >= 3 and ft_score >= 3:
        recommendation = "Both: RAG + Fine-tuning"
    elif rag_score > ft_score:
        recommendation = "RAG"
    elif ft_score > rag_score:
        recommendation = "Fine-tuning"
    else:
        recommendation = "Start with prompting/RAG prototype, rồi đo chất lượng trước khi fine-tune"

    return {
        "rag_score": rag_score,
        "fine_tuning_score": ft_score,
        "recommendation": recommendation,
        "warnings": warnings,
    }

# Ví dụ 1: chatbot hỏi đáp tài liệu nội bộ thay đổi thường xuyên
choose_augmentation(
    needs_private_knowledge=True,
    data_changes_often=True,
    needs_strict_output_format=False,
    needs_new_skill_or_behavior=False,
    has_labeled_training_data=False,
    question_requires_global_aggregation=False,
)

In [ ]:
# TODO: Thử sửa case này.
# Ví dụ 2: phân loại ticket support vào đúng team, output phải đúng JSON schema.
choose_augmentation(
    needs_private_knowledge=False,
    data_changes_often=False,
    needs_strict_output_format=True,
    needs_new_skill_or_behavior=True,
    has_labeled_training_data=True,
    question_requires_global_aggregation=False,
)

## 4. Chi phí: vì sao fine-tuning/self-hosting không chỉ là chuyện train một lần?

Fine-tuning thường kéo theo:

- training pipeline;
- model registry;
- validation;
- deployment/canary/A-B test;
- monitoring;
- retraining khi dữ liệu drift;
- chi phí chuyên môn MLOps/ML Engineering.

Ta mô phỏng nhanh số tham số, memory, và chi phí GPU theo rule-of-thumb đơn giản.

In [ ]:
def model_memory_gb(num_params_billions, bytes_per_param=2):
    """Approx memory chỉ riêng weights."""
    return num_params_billions * 1e9 * bytes_per_param / 1024**3


def training_memory_gb(num_params_billions, bytes_per_param=2, overhead_multiplier=4):
    return model_memory_gb(num_params_billions, bytes_per_param) * overhead_multiplier


def serving_memory_gb(num_params_billions, bytes_per_param=2, overhead_multiplier=2):
    return model_memory_gb(num_params_billions, bytes_per_param) * overhead_multiplier

models = pd.DataFrame({
    "model": ["BERT-large-ish 0.345B", "Llama-style 7B", "Llama-style 13B", "Llama-style 70B"],
    "params_B": [0.345, 7, 13, 70],
})

models["weights_memory_GB_fp16"] = models["params_B"].apply(lambda x: model_memory_gb(x, bytes_per_param=2))
models["train_memory_GB_rule_of_thumb"] = models["params_B"].apply(training_memory_gb)
models["serve_memory_GB_rule_of_thumb"] = models["params_B"].apply(serving_memory_gb)
models.round(2)

In [ ]:
ax = models.plot(
    x="model",
    y=["weights_memory_GB_fp16", "train_memory_GB_rule_of_thumb", "serve_memory_GB_rule_of_thumb"],
    kind="bar",
    figsize=(10, 4),
)
ax.set_title("Memory tăng rất nhanh khi model lớn hơn")
ax.set_ylabel("Approx GB")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

### Ước lượng chi phí API token

Hosted API có thể đắt nếu volume rất lớn, nhưng lại giảm đáng kể gánh nặng vận hành. Cell dưới đây cho bạn tự đổi giá token và traffic.

In [ ]:
def api_daily_cost(num_requests_per_day, input_tokens_per_request, output_tokens_per_request,
                   input_price_per_1k, output_price_per_1k):
    input_cost = num_requests_per_day * input_tokens_per_request / 1000 * input_price_per_1k
    output_cost = num_requests_per_day * output_tokens_per_request / 1000 * output_price_per_1k
    return input_cost + output_cost

traffic_scenarios = pd.DataFrame({
    "requests_per_day": [1_000, 10_000, 100_000, 1_000_000],
    "input_tokens": [1000, 1000, 1000, 1000],
    "output_tokens": [300, 300, 300, 300],
})

# TODO: thay giá theo model/API bạn đang cân nhắc.
traffic_scenarios["daily_cost_usd"] = traffic_scenarios.apply(
    lambda row: api_daily_cost(
        row.requests_per_day,
        row.input_tokens,
        row.output_tokens,
        input_price_per_1k=0.0015,
        output_price_per_1k=0.0020,
    ),
    axis=1,
)
traffic_scenarios["monthly_cost_usd_30d"] = traffic_scenarios["daily_cost_usd"] * 30
traffic_scenarios

## 5. RAG toy demo: vector search hoạt động thế nào?

Ta sẽ tự làm embedding cực đơn giản bằng bag-of-words để nhớ pipeline RAG:

1. Có documents.
2. Chunk/index documents thành vector.
3. Query cũng biến thành vector.
4. Dùng cosine similarity để retrieve documents gần nhất.
5. Đưa documents đó vào prompt.

Lưu ý: demo này dùng embedding rất đơn giản để minh họa. Embedding LLM thật sẽ tốt hơn nhiều.

In [ ]:
documents = [
    "RAG retrieves relevant documents and injects them into the prompt for the language model.",
    "Fine-tuning updates model weights or adapter weights so the model learns a task-specific behavior.",
    "LoRA freezes the original model weights and trains small low-rank adapter matrices.",
    "Vector databases store embeddings and support similarity search with metrics such as cosine similarity.",
    "Questions requiring global aggregation, such as finding the earliest document, are not ideal for pure vector search.",
    "Catastrophic forgetting can happen when a model is repeatedly fine-tuned and loses older capabilities.",
]

import re

def tokenize(text):
    return re.findall(r"[a-zA-Z]+", text.lower())

vocab = sorted(set(token for doc in documents for token in tokenize(doc)))
word_to_idx = {word: i for i, word in enumerate(vocab)}

def bow_embed(text):
    vector = np.zeros(len(vocab))
    for token in tokenize(text):
        if token in word_to_idx:
            vector[word_to_idx[token]] += 1
    norm = np.linalg.norm(vector)
    return vector / norm if norm > 0 else vector

embeddings = np.vstack([bow_embed(doc) for doc in documents])


def cosine_search(query, top_k=3):
    q = bow_embed(query)
    scores = embeddings @ q
    order = np.argsort(scores)[::-1][:top_k]
    return pd.DataFrame({
        "rank": range(1, len(order) + 1),
        "score": scores[order],
        "document": [documents[i] for i in order],
    })

cosine_search("How can an LLM use private company documents without retraining?", top_k=3)

## 6. Vấn đề RAG #1: câu hỏi không luôn giống câu trả lời về mặt semantic

Một câu hỏi có thể dùng từ khác với tài liệu chứa câu trả lời. Khi đó vector search có thể retrieve sai.

Kỹ thuật **HyDE** giải quyết bằng cách:

1. LLM tạo một câu trả lời giả định/hypothetical answer.
2. Embed câu trả lời giả định đó.
3. Dùng embedding này để search.

Trong demo dưới đây, ta mô phỏng HyDE bằng cách tự viết hypothetical answer.

In [ ]:
query = "How do we avoid changing the model when adding new knowledge?"
print("Search bằng query gốc:")
display(cosine_search(query, top_k=3))

hyde_answer = "Use retrieval augmented generation to retrieve relevant documents and inject them into the prompt without updating model weights."
print("Search bằng hypothetical answer kiểu HyDE:")
display(cosine_search(hyde_answer, top_k=3))

## 7. Vấn đề RAG #2: chunk quá lớn làm loãng ý nghĩa

Nếu một chunk chứa quá nhiều ý khác nhau, similarity search có thể bị nhiễu.

Ta so sánh:

- chunk nhỏ: mỗi chunk một ý rõ ràng;
- chunk lớn: nhiều ý bị gộp lại.

In [ ]:
small_chunks = [
    "RAG retrieves documents for the prompt.",
    "Fine-tuning changes model behavior by training weights.",
    "LoRA trains small adapter matrices.",
    "Vector databases perform similarity search.",
]

large_chunks = [
    "RAG retrieves documents for the prompt. Fine-tuning changes model behavior by training weights. LoRA trains small adapter matrices. Vector databases perform similarity search.",
]

print("Small chunks giữ từng ý riêng biệt:")
for i, chunk in enumerate(small_chunks, 1):
    print(f"{i}. {chunk}")

print("\nLarge chunk gộp nhiều ý:")
print(large_chunks[0])

## 8. Vấn đề RAG #3: không phù hợp cho câu hỏi cần aggregate toàn database

Vector search tìm vài chunk giống query nhất. Nhưng câu hỏi như:

- “Document nào sớm nhất?”
- “Tổng doanh thu là bao nhiêu?”
- “Khách hàng nào mua nhiều nhất?”

cần scan/aggregate toàn bộ dữ liệu, không chỉ retrieve top-k chunk.

In [ ]:
records = pd.DataFrame({
    "doc_id": ["A", "B", "C", "D"],
    "date": pd.to_datetime(["2024-03-05", "2023-11-20", "2025-01-10", "2022-07-01"]),
    "text": [
        "RAG overview document",
        "Fine-tuning pipeline document",
        "Vector database document",
        "Model monitoring document",
    ]
})

# Cách đúng cho câu hỏi aggregate: dùng structured query/dataframe operation.
earliest = records.sort_values("date").head(1)
earliest

## 9. Vấn đề RAG #4: thứ tự document trong prompt quan trọng

LLM thường chú ý mạnh hơn đến thông tin ở đầu/cuối context, và có thể bỏ sót phần giữa. Một mẹo là đặt các document quan trọng nhất xen kẽ ở đầu và cuối.

Cell dưới đây reorder danh sách theo kiểu:

- relevance cao nhất lên đầu;
- relevance cao nhì xuống cuối;
- relevance tiếp theo lên gần đầu;
- relevance tiếp theo xuống gần cuối.

In [ ]:
def edge_reorder(items):
    """items đã sort từ relevant nhất đến ít relevant nhất."""
    result = []
    left = []
    right = []
    for i, item in enumerate(items):
        if i % 2 == 0:
            left.append(item)
        else:
            right.insert(0, item)
    return left + right

retrieved_docs = [
    "doc_1_score_0.95",
    "doc_2_score_0.91",
    "doc_3_score_0.87",
    "doc_4_score_0.76",
    "doc_5_score_0.61",
]

pd.DataFrame({
    "original_order": retrieved_docs,
    "edge_reordered": edge_reorder(retrieved_docs),
})

## 10. Fine-tuning pipeline checklist

Nếu chọn fine-tuning, đừng chỉ nghĩ đến training. Hãy nhớ checklist pipeline.

In [ ]:
finetuning_checklist = pd.DataFrame([
    ["Training data", "Có input/output đúng task, đủ chất lượng, sạch lỗi nhãn"],
    ["Validation set", "Đo task mới và kiểm tra model không mất năng lực cũ"],
    ["Model registry", "Version model gốc, model fine-tuned, metadata"],
    ["LoRA/QLoRA registry", "Version adapter weights nếu dùng parameter-efficient fine-tuning"],
    ["Deployment pipeline", "Canary, A/B test, rollback plan"],
    ["Monitoring", "Theo dõi chất lượng, latency, cost, drift"],
    ["Retraining plan", "Khi dữ liệu đổi, cần kế hoạch train lại an toàn"],
], columns=["component", "why_it_matters"])
finetuning_checklist

## 11. RAG pipeline checklist

Nếu chọn RAG, trọng tâm là retrieval quality và data pipeline.

In [ ]:
rag_checklist = pd.DataFrame([
    ["Data ingestion", "Kéo dữ liệu mới từ docs/wiki/db vào pipeline"],
    ["Chunking", "Chia tài liệu thành đoạn đủ nhỏ, mỗi chunk ít ý chính"],
    ["Embedding service", "Biến chunk và query thành vector"],
    ["Vector database", "Index/search embeddings"],
    ["Retriever", "Chọn top-k chunk liên quan; có thể rerank"],
    ["Prompt builder", "Nhét context vào prompt theo thứ tự tốt"],
    ["Evaluation", "Đo retrieval hit-rate, groundedness, hallucination"],
], columns=["component", "why_it_matters"])
rag_checklist

## 12. Mini quiz để tự nhớ bài

Chạy cell dưới đây, đọc câu hỏi và tự trả lời trước khi xem đáp án.

In [ ]:
quiz = [
    {
        "question": "Muốn chatbot trả lời theo tài liệu nội bộ cập nhật hằng ngày, nên bắt đầu với gì?",
        "answer": "RAG, vì dữ liệu thay đổi thường xuyên và cần truy xuất kiến thức ngoài model.",
    },
    {
        "question": "Muốn model luôn output đúng JSON schema cho task routing ticket, hướng nào đáng cân nhắc?",
        "answer": "Fine-tuning, nếu có dữ liệu labeled tốt; cũng có thể kết hợp prompt/validation.",
    },
    {
        "question": "RAG có tốt cho câu hỏi 'tổng số invoice tháng này là bao nhiêu' không?",
        "answer": "Không nên dùng RAG thuần; cần query/aggregate trên database có cấu trúc.",
    },
    {
        "question": "Catastrophic forgetting liên quan đến hướng nào?",
        "answer": "Fine-tuning, đặc biệt khi fine-tune lặp lại làm model quên năng lực cũ.",
    },
    {
        "question": "HyDE giúp xử lý vấn đề gì trong RAG?",
        "answer": "Câu hỏi không giống câu trả lời về semantic; tạo hypothetical answer rồi embed để search tốt hơn.",
    },
]

for i, item in enumerate(quiz, 1):
    print(f"Q{i}: {item['question']}")
    print(f"A{i}: {item['answer']}\n")

## 13. Cheat sheet cuối cùng

| Nếu bạn cần... | Nên chọn |
|---|---|
| Thêm kiến thức riêng, tài liệu mới, dữ liệu cập nhật liên tục | RAG |
| Model học task/hành vi/format ổn định | Fine-tuning |
| Vừa cần kiến thức ngoài, vừa cần hành vi rất đặc thù | RAG + Fine-tuning |
| Trả lời câu hỏi cần tính tổng/min/max/count trên toàn DB | Structured query/tool, không dùng RAG thuần |
| Giảm chi phí vận hành ban đầu | Hosted API + RAG/prototype trước |

**Câu nhớ nhanh:**

> RAG = “cho model đọc tài liệu đúng lúc”.  
> Fine-tuning = “dạy model một thói quen/kỹ năng mới”.